# Advanced Regular Expressions

This notebook builds on the beginner regex notebook and covers more powerful features:
- Lookaheads and lookbehinds
- Non-greedy matching
- Backreferences
- Named groups
- Compiled patterns
- Real-world patterns: log parsing, data extraction

## 1. Recap: Core Patterns

In [ ]:
import re

text = 'Call 555-1234 or email support@example.com for help'

# Phone pattern
phones = re.findall(r'\d{3}-\d{4}', text)
print('Phones:', phones)

# Email pattern
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', text)
print('Emails:', emails)

## 2. Greedy vs Non-Greedy

By default, quantifiers are **greedy** — they match as much as possible.  
Add `?` to make them **non-greedy** (lazy) — match as little as possible.

In [ ]:
import re

html = '<p>Hello</p><p>World</p>'

# Greedy — matches from first < to last >
print('Greedy:', re.findall(r'<p>.*</p>', html))

# Non-greedy — matches each <p>...</p> separately
print('Non-greedy:', re.findall(r'<p>.*?</p>', html))

# Another example: string literals in code
code = 'x = "hello" + "world"'
print('Greedy strings:', re.findall(r'".*"', code))    # one match
print('Non-greedy strings:', re.findall(r'".*?"', code))  # two matches

## 3. Lookahead and Lookbehind

**Lookahead/lookbehind** match based on surrounding context, WITHOUT including that context in the match.

| Syntax | Name | Matches |
|--------|------|--------|
| `(?=...)` | Positive lookahead | position followed by `...` |
| `(?!...)` | Negative lookahead | position NOT followed by `...` |
| `(?<=...)` | Positive lookbehind | position preceded by `...` |
| `(?<!...)` | Negative lookbehind | position NOT preceded by `...` |

In [ ]:
import re

# Positive lookahead: \d+ followed by ' kg'
weights = re.findall(r'\d+(?= kg)', 'I weigh 70 kg and my bag is 15 kg')
print('Weights:', weights)  # ['70', '15'] — without ' kg'

# Negative lookahead: number NOT followed by ' kg'
text = 'Price: £100. Weight: 70 kg. Count: 5'
non_weights = re.findall(r'\d+(?! kg)', text)
print('Non-weights:', non_weights)

# Positive lookbehind: number preceded by '£'
prices = re.findall(r'(?<=£)\d+', 'Prices: £10, £25, €30, £100')
print('GBP prices:', prices)  # ['10', '25', '100'] — without '£'

# Negative lookbehind: number NOT preceded by '£'
non_gbp = re.findall(r'(?<!£)\d+', 'Items: 5, £10, 3, £25')
print('Non-GBP:', non_gbp)

## 4. Named Groups and Backreferences

In [ ]:
import re

# Named groups (?P<name>pattern)
log = '2024-01-15 14:30:45 ERROR Database connection failed'

pattern = re.compile(
    r'(?P<date>\d{4}-\d{2}-\d{2}) '
    r'(?P<time>\d{2}:\d{2}:\d{2}) '
    r'(?P<level>\w+) '
    r'(?P<message>.+)'
)

m = pattern.match(log)
if m:
    print(f"Date: {m.group('date')}")
    print(f"Time: {m.group('time')}")
    print(f"Level: {m.group('level')}")
    print(f"Message: {m.group('message')}")
    print(f"Dict: {m.groupdict()}")

In [ ]:
import re

# Backreferences — match the same text as a previous group
# Find duplicate words
text = 'the the quick brown fox fox jumps over the lazy dog'
duplicates = re.findall(r'\b(\w+) \1\b', text)
print('Duplicate words:', duplicates)  # ['the', 'fox']

# Find matching HTML tags
html = '<h1>Title</h1><p>Para</p><div>Content</div>'
matched_tags = re.findall(r'<(\w+)>.*?</\1>', html)
print('Matched tags:', matched_tags)

## 5. `re.compile()` and Performance

In [ ]:
import re
import time

# Compile once for reuse
email_re = re.compile(r'^[\w.+-]+@[\w-]+\.[\w.]{2,}$', re.IGNORECASE)

test_emails = [
    'user@example.com',
    'bad-email',
    'user@.com',
    'ALICE@EXAMPLE.ORG',
    'firstname.lastname@company.co.uk',
    '@nodomain.com',
]

print('Email Validation:')
for email in test_emails:
    valid = bool(email_re.match(email))
    status = 'VALID  ' if valid else 'INVALID'
    print(f'  {status}: {email}')

## 6. `re.VERBOSE` — Readable Patterns

In [ ]:
import re

# Complex URL pattern — hard to read
url_compact = re.compile(r'(https?)://([\w.-]+)((?:/[\w./?=&#%+-]*)?)(?:#([\w-]*))?')

# Same pattern — readable with VERBOSE
url_verbose = re.compile(r'''
    (?P<scheme>https?)   # protocol: http or https
    ://                  # separator
    (?P<domain>[\w.-]+) # domain name
    (?P<path>            # optional path
        /[\w./?=&#%+-]*
    )?                   
    (?:[#]               # optional fragment
        (?P<fragment>[\w-]*)
    )?
''', re.VERBOSE)

urls = [
    'https://www.example.com/path/to/page?q=1&page=2',
    'http://api.service.io/v2/users',
    'https://docs.python.org/3/library/re.html#re.compile',
]

for url in urls:
    m = url_verbose.match(url)
    if m:
        d = m.groupdict()
        print(f'URL: {url}')
        print(f'  scheme={d["scheme"]}, domain={d["domain"]}')
        if d.get('fragment'):
            print(f'  fragment={d["fragment"]}')
        print()

## 7. Common Patterns Library

In [ ]:
import re

PATTERNS = {
    'email': r'^[\w.+-]+@[\w-]+\.[\w.]{2,}$',
    'url': r'^https?://[\w.-]+(?:/[\w./?=&#%+-]*)?$',
    'ipv4': r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$',
    'date_iso': r'^\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])$',
    'credit_card': r'^(?:4\d{15}|5[1-5]\d{14}|3[47]\d{13})$',
    'phone_uk': r'^(?:\+44|0)\s?\d{4}\s?\d{6}$',
    'postcode_uk': r'^[A-Z]{1,2}\d[A-Z\d]? ?\d[A-Z]{2}$',
    'hex_color': r'^#(?:[0-9A-Fa-f]{6}|[0-9A-Fa-f]{3})$',
}

compiled = {k: re.compile(v) for k, v in PATTERNS.items()}

test_cases = {
    'email':       ['user@example.com', 'bad@email'],
    'ipv4':        ['192.168.1.1', '999.0.0.1'],
    'date_iso':    ['2024-01-15', '2024-13-01'],
    'hex_color':   ['#FF5733', '#XYZ'],
}

for pattern_name, values in test_cases.items():
    print(f'{pattern_name.upper()}:')
    for v in values:
        valid = bool(compiled[pattern_name].match(v))
        print(f'  {"✓" if valid else "✗"} {v}')
    print()

## Mini-Project: Log File Parser

In [ ]:
import re
from collections import Counter, defaultdict

# Sample log data
log_data = '''
2024-01-15 08:00:01 INFO Server started on port 8080
2024-01-15 08:00:15 DEBUG Loading configuration from config.yaml
2024-01-15 08:01:02 INFO New connection from 192.168.1.101
2024-01-15 08:01:05 INFO New connection from 192.168.1.102
2024-01-15 08:02:33 WARNING High memory usage: 85%
2024-01-15 08:03:11 ERROR Failed to connect to database: Connection refused
2024-01-15 08:03:12 INFO Retrying connection (attempt 1/3)
2024-01-15 08:03:14 INFO Retrying connection (attempt 2/3)
2024-01-15 08:03:16 INFO Database connected successfully
2024-01-15 08:05:22 ERROR Invalid request from 10.0.0.5: Missing auth token
2024-01-15 08:10:00 WARNING Disk usage above 80%: /var/log partition
2024-01-15 08:15:45 INFO Request processed: GET /api/users (200 OK, 45ms)
2024-01-15 08:16:02 INFO Request processed: POST /api/users (201 Created, 120ms)
2024-01-15 08:20:00 ERROR Unhandled exception in request handler
'''.strip()


def parse_logs(log_text):
    entry_pattern = re.compile(
        r'(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) '
        r'(?P<level>INFO|DEBUG|WARNING|ERROR|CRITICAL) '
        r'(?P<message>.+)'
    )
    ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
    request_pattern = re.compile(r'(GET|POST|PUT|DELETE|PATCH) (/\S*) \((\d{3})')
    
    entries = []
    level_counts = Counter()
    ips = set()
    requests_log = []
    
    for line in log_text.strip().split('\n'):
        m = entry_pattern.match(line)
        if not m:
            continue
        
        entry = m.groupdict()
        entries.append(entry)
        level_counts[entry['level']] += 1
        
        # Extract IPs
        for ip in ip_pattern.findall(entry['message']):
            ips.add(ip)
        
        # Extract HTTP requests
        req_m = request_pattern.search(entry['message'])
        if req_m:
            requests_log.append({
                'method': req_m.group(1),
                'path': req_m.group(2),
                'status': req_m.group(3)
            })
    
    return entries, level_counts, ips, requests_log


entries, level_counts, ips, reqs = parse_logs(log_data)

print('=== LOG ANALYSIS ===')
print(f'Total entries: {len(entries)}')
print('\nBy level:')
for level in ['ERROR', 'WARNING', 'INFO', 'DEBUG']:
    count = level_counts.get(level, 0)
    bar = '█' * count
    print(f'  {level:8} {count:3}  {bar}')

print(f'\nUnique IPs seen: {ips}')

print('\nHTTP Requests:')
for r in reqs:
    print(f"  {r['method']:6} {r['path']:20} → {r['status']}")

errors = [e for e in entries if e['level'] == 'ERROR']
print(f'\nError messages:')
for e in errors:
    print(f"  {e['timestamp']}  {e['message']}")